In [2]:
attendance_log_df=spark.read.format('parquet').load('abfss://project5_workspace@onelake.dfs.fabric.microsoft.com/lakehouse5.Lakehouse/Files/bronze/attendance_logs.parquet')
display(attendance_log_df)


StatementMeta(, e4ea1dd0-2189-4d37-91ff-bdf632f820c1, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 59289066-3767-4137-b5c1-537dd2374c24)

In [3]:
from pyspark.sql.functions import *
attendance_log_silver_df=attendance_log_df.withColumnRenamed('Emp ID','Employee_id')\
.withColumn('AbsenceCount',col('AbsenceCount').cast('int'))\
.withColumn('Month',to_date('Month','yyyy-MM'))\
.withColumnRenamed('AbsenceCount','Absence_days')
attendance_log_silver_df.write.format('delta').save('Files/silver/attendance_log')

StatementMeta(, e4ea1dd0-2189-4d37-91ff-bdf632f820c1, 5, Finished, Available, Finished, False)

In [4]:
employee_details_df=spark.read.format('parquet').load('abfss://project5_workspace@onelake.dfs.fabric.microsoft.com/lakehouse5.Lakehouse/Files/bronze/employee_details.parquet')
display(employee_details_df)

StatementMeta(, e4ea1dd0-2189-4d37-91ff-bdf632f820c1, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4ecbe315-a03c-49f1-89a8-a97ca32210e8)

In [5]:
from pyspark.sql.functions import *
employee_detail_silver_df=employee_details_df.withColumnRenamed('Emp ID','employee_id')\
.withColumn('Name',initcap(trim('Name')))\
.withColumn('Gender',when(lower(col('Gender')).isin('male','m'),'Male').otherwise('Female'))\
.withColumn('Join Date',to_date('Join Date','MM/dd/yyyy'))\
.withColumnRenamed('Join Date','join_date')
employee_detail_silver_df.write.format('delta').save('Files/silver/employee_detail')

StatementMeta(, e4ea1dd0-2189-4d37-91ff-bdf632f820c1, 7, Finished, Available, Finished, False)

In [6]:
employee_salary_df=spark.read.format('parquet').option('header',True).load('abfss://project5_workspace@onelake.dfs.fabric.microsoft.com/lakehouse5.Lakehouse/Files/bronze/employee_salaries.xlsx.parquet')
employee_salary_df=employee_salary_df.toDF('employee_id','salary')
employee_salary_silver_df=employee_salary_df.filter(col('employee_id')!='Emp ID')\
.withColumn('salary',col('salary').cast('double'))
employee_salary_silver_df.write.format('delta').save('Files/silver/employee_salary')

StatementMeta(, e4ea1dd0-2189-4d37-91ff-bdf632f820c1, 8, Finished, Available, Finished, False)

In [7]:
performance_review_df=spark.read.format('parquet').load('abfss://project5_workspace@onelake.dfs.fabric.microsoft.com/lakehouse5.Lakehouse/Files/bronze/performance_reviews.parquet')
display(performance_review_df)

StatementMeta(, e4ea1dd0-2189-4d37-91ff-bdf632f820c1, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d718e633-759a-4388-965d-fcfd5df81685)

In [8]:
performance_review_silver_df=performance_review_df.withColumnRenamed('Emp ID','Employee_id')\
.withColumn('Review_date',to_date('Review_date','yyyy-MM-dd'))\
.withColumn('Rating',col('Rating').cast('double'))
performance_review_silver_df.write.format('delta').save('Files/silver/performance_review')

StatementMeta(, e4ea1dd0-2189-4d37-91ff-bdf632f820c1, 10, Finished, Available, Finished, False)

In [9]:
resignation_log_df=spark.read.format('parquet').load('abfss://project5_workspace@onelake.dfs.fabric.microsoft.com/lakehouse5.Lakehouse/Files/bronze/resignation_logs.parquet')
display(resignation_log_df)

StatementMeta(, e4ea1dd0-2189-4d37-91ff-bdf632f820c1, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1f80cb3b-f04c-4875-ad72-c549461be8ec)

In [10]:
resignation_silver_log_df=resignation_log_df.withColumnRenamed('Emp ID','Employee_id')\
.withColumn('ResignationDate',to_date('ResignationDate','yyyy-MM-dd'))\
.withColumnRenamed('ResignationDate','Resignation_date')\
.withColumn('ResignationReason',initcap(trim('ResignationReason')))\
.withColumnRenamed('ResignationReason','Resignation_reason')
resignation_silver_log_df.write.format('delta').save('Files/silver/resignation_log')

StatementMeta(, e4ea1dd0-2189-4d37-91ff-bdf632f820c1, 12, Finished, Available, Finished, False)

In [8]:
attendance_log_silver_df1=spark.read.format('delta').load('abfss://project5_workspace@onelake.dfs.fabric.microsoft.com/lakehouse5.Lakehouse/Files/silver/attendance_log')
employee_detail_silver_df1=spark.read.format('delta').load('abfss://project5_workspace@onelake.dfs.fabric.microsoft.com/lakehouse5.Lakehouse/Files/silver/employee_detail')
employee_salary_silver_df1=spark.read.format('delta').load('abfss://project5_workspace@onelake.dfs.fabric.microsoft.com/lakehouse5.Lakehouse/Files/silver/employee_salary')
performance_review_silver_df1=spark.read.format('delta').load('abfss://project5_workspace@onelake.dfs.fabric.microsoft.com/lakehouse5.Lakehouse/Files/silver/performance_review')
resignation_log_silver_df1=spark.read.format('delta').load('abfss://project5_workspace@onelake.dfs.fabric.microsoft.com/lakehouse5.Lakehouse/Files/silver/resignation_log')

StatementMeta(, fc067cd8-df96-4b90-9075-88cb95b95aaf, 10, Finished, Available, Finished, False)

In [9]:
from pyspark.sql.functions import *
employee_final_df=employee_detail_silver_df1.join(employee_salary_silver_df1,employee_detail_silver_df1.employee_id==employee_salary_silver_df1.employee_id,how='left')\
.join(attendance_log_silver_df1,employee_detail_silver_df1.employee_id==attendance_log_silver_df1.Employee_id,how='left')\
.join(performance_review_silver_df1,employee_detail_silver_df1.employee_id==performance_review_silver_df1.Employee_id)\
.join(resignation_log_silver_df1,employee_detail_silver_df1.employee_id==resignation_log_silver_df1.Employee_id,how='left')\
.drop(employee_salary_silver_df1.employee_id,attendance_log_silver_df1.Employee_id,performance_review_silver_df1.Employee_id,resignation_log_silver_df1.Employee_id)
employee_final_df=employee_final_df.withColumn('total_experience',when(col('Resignation_date').isNotNull(),datediff('Resignation_date','join_date'))\
.otherwise(datediff(current_date(),'join_date'))
)\
.withColumn('total_experience_years',floor(col('total_experience')/365))\
.withColumn('total_experience_months',floor((col('total_experience')-(365*col('total_experience_years')))/30))\
.withColumn('total_experience_days',col('total_experience')-(365*col('total_experience_years'))-(30*col('total_experience_months')))
employee_final_df.write.format('delta').save('Files/gold/employee_final_df')

StatementMeta(, fc067cd8-df96-4b90-9075-88cb95b95aaf, 11, Finished, Available, Finished, False)

In [11]:
employee_final_df.write.format('delta').saveAsTable('employee_gold_final_df')

StatementMeta(, fc067cd8-df96-4b90-9075-88cb95b95aaf, 13, Finished, Available, Finished, False)

In [10]:
from pyspark.sql.functions import *
employee_gold_df=spark.read.format('delta').load('abfss://project5_workspace@onelake.dfs.fabric.microsoft.com/lakehouse5.Lakehouse/Files/gold/employee_final_df')
employee_gold_df=employee_gold_df.withColumn('total_experience',when(col('Resignation_date').isNotNull(),datediff('Resignation_date','join_date'))\
.otherwise(datediff(current_date(),'join_date'))
)\
.withColumn('total_experience_years',floor(col('total_experience')/365))\
.withColumn('total_experience_months',floor((col('total_experience')-(365*col('total_experience_years')))/30))\
.withColumn('total_experience_days',col('total_experience')-(365*col('total_experience_years'))-(30*col('total_experience_months')))
employee_gold_metrics_df=employee_gold_df.filter(col('Resignation_reason').isNotNull()).groupBy('Department','Gender','Resignation_reason').agg(count('employee_id').alias('total_employees'),avg('salary').alias('avg_salary'))
employee_gold_metrics_df.write.format('delta').save('Files/gold/employee_gold_metrics_df')

StatementMeta(, fc067cd8-df96-4b90-9075-88cb95b95aaf, 12, Finished, Available, Finished, False)